# 🕷️ AIOS Scraper Farm (Playwright)

Высокоскоростной скрапинг с **чистых IP Google** через гигабитный канал Colab.
Мониторинг: аирдропы, CryptoPanic, Freelancehunt, DEX-пулы, новости.

Задания можно создавать на VPS: `python scripts/dispatch_colab_scrape.py create ...`.

Результаты сохраняются в файл и переносятся на VPS для ингеста (`result_ingest.py`).

In [ ]:
!pip install -q playwright pandas
!playwright install chromium
from playwright.sync_api import sync_playwright
import json, re
print('✅ Playwright установлен')

In [ ]:
# === ЯЧЕЙКА 2: Конфигурация задачи ===
JOB = {
    'source': 'cryptopanic',   # airdrops | cryptopanic | freelancehunt | dex | news
    'urls': ['https://cryptopanic.com/news/'],
    'max_pages': 2,
}
print('Задание:', JOB)

In [ ]:
# === ЯЧЕЙКА 3: Скрапинг через Playwright ===
results = []
with sync_playwright() as p:
    browser = p.chromium.launch(headless=True, args=['--no-sandbox'])
    page = browser.new_page()
    for url in JOB['urls']:
        page.goto(url, wait_until='domcontentloaded', timeout=60000)
        page.wait_for_timeout(3000)
        items = page.eval_on_selector_all(
            'a, h1, h2, h3, article, .title, [class*=title]',
            """els => els.slice(0,60).map(e => ({
                tag: e.tagName,
                text: (e.innerText||'').trim().slice(0,250),
                href: e.href||''
            }))""")
        results.extend(items)
    browser.close()
print('✅ Собрано элементов:', len(results))

In [ ]:
# === ЯЧЕЙКА 4: Нормализация и сохранение ===
norm = []
seen = set()
for it in results:
    t = (it.get('text') or '').strip()
    if not t or len(t) < 5:
        continue
    key = (it.get('href') or t)
    if key in seen:
        continue
    seen.add(key)
    norm.append({'source': JOB['source'], 'title': t, 'url': it.get('href')})
print('Нормализовано:', len(norm))

with open('scrape_results.json', 'w', encoding='utf-8') as f:
    json.dump(norm, f, ensure_ascii=False, indent=2)
print('✅ Сохранено: scrape_results.json')
print('Скачайте файл на VPS и запустите:')
print('  python scripts/dispatch_colab_scrape.py create --source <SOURCE> --target <URL>')
print('  python aios_core/scraping/result_ingest.py --source <SOURCE> --input scrape_results.json')